### 복습
- 가전 폴더 안에 모든 데이터파일을 로드해서 하나의 데이터프레임으로 생성
- 감정에 대한 데이터들이 3개 분류 -> 2개의 분류로 변경(부정, 중립 -> 부정)
- 감정 데이터가 없는 데이터들은 따로 저장
- train, test의 비율은 8:2
- Dataset을 기존의 Dataset 구성과 같이 작업
- RawText 데이터를 이용하여 감정분석 모델을 생성
- SBERT 모델을 이용하여 임베딩
- 다중퍼셉트론의 모델을 이용하여 감정 분석 (Linear -> ReLU -> DropOut -> Linear)
- 검증 데이터를 이용하여 정확도와 f1_score 확인
- 감정 데이터가 없는 RawText에서 sample(10)를 출력하여 감정 예측

- 다중퍼셉트론 모델이 아닌 머신러닝 모델 (SVC)을 이용하여 감정 분석 예측

In [1]:
import os
import re
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sentence_transformers import SentenceTransformer

c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# 문자 정규화 함수 정의
def normalize(text):
    text = re.sub(r'[^가-힣0-9a-zA-Z\s\.]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [12]:
# 파일의 목록을 로드 -> 목록을 기준으로 데이터를 로드 -> 단순 결합
file_path = '../data/가전/'
file_list = os.listdir(file_path)
file_list

['3-1.영상음향가전(76).json',
 '3-1.영상음향가전(77).json',
 '3-1.영상음향가전(78).json',
 '3-1.영상음향가전(79).json',
 '3-1.영상음향가전(80).json',
 '3-1.영상음향가전(81).json',
 '3-1.영상음향가전(82).json',
 '3-1.영상음향가전(83).json',
 '3-1.영상음향가전(84).json',
 '3-1.영상음향가전(85).json',
 '3-1.영상음향가전(86).json',
 '3-1.영상음향가전(87).json',
 '3-1.영상음향가전(88).json',
 '3-2.생활미용욕실가전(128).json',
 '3-2.생활미용욕실가전(129).json',
 '3-2.생활미용욕실가전(130).json',
 '3-2.생활미용욕실가전(131).json',
 '3-2.생활미용욕실가전(132).json',
 '3-2.생활미용욕실가전(133).json',
 '3-2.생활미용욕실가전(134).json',
 '3-2.생활미용욕실가전(135).json',
 '3-2.생활미용욕실가전(136).json',
 '3-2.생활미용욕실가전(137).json',
 '3-2.생활미용욕실가전(138).json',
 '3-2.생활미용욕실가전(139).json',
 '3-2.생활미용욕실가전(140).json',
 '3-3.주방가전(127).json',
 '3-3.주방가전(128).json',
 '3-3.주방가전(129).json',
 '3-3.주방가전(130).json',
 '3-3.주방가전(131).json',
 '3-3.주방가전(132).json',
 '3-3.주방가전(133).json',
 '3-3.주방가전(134).json',
 '3-3.주방가전(135).json',
 '3-3.주방가전(136).json',
 '3-3.주방가전(137).json',
 '3-3.주방가전(138).json',
 '3-3.주방가전(139).json',
 '3-4.계절가전(126).json',
 '3-4.계절가전(127)

In [13]:
# 로드한 데이터프레임을 누저긍로 결합하기 위해 빈 데이터프레임 생성
df = pd.DataFrame()

for file in file_list:
    # file : 파일명
    data = pd.read_json(file_path + file)
    # df에 단순 행 결합 -> df에 다시 대입
    df = pd.concat([df, data], axis=0)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4056 entries, 0 to 99
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            4056 non-null   int64  
 1   RawText          4056 non-null   object 
 2   Source           4056 non-null   object 
 3   Domain           4056 non-null   object 
 4   MainCategory     4056 non-null   object 
 5   ProductName      4056 non-null   object 
 6   ReviewScore      4056 non-null   int64  
 7   Syllable         4056 non-null   int64  
 8   Word             4056 non-null   int64  
 9   RDate            4056 non-null   int64  
 10  GeneralPolarity  3678 non-null   float64
 11  Aspects          4056 non-null   object 
dtypes: float64(1), int64(5), object(6)
memory usage: 411.9+ KB


In [ ]:
# 필요한 컬럼을 제외하고 나머지 컬럼을 무시
df = df[['RawText', 'GeneralPolarity']]

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4056 entries, 0 to 99
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RawText          4056 non-null   object 
 1   GeneralPolarity  3678 non-null   float64
dtypes: float64(1), object(1)
memory usage: 95.1+ KB


In [16]:
df.rename(columns={'GeneralPolarity' : 'label'}, inplace=True)

In [17]:
# RawText의 문자 정규화 사용
df['RawText'] = df['RawText'].map(normalize)

In [18]:
# RawText에서 길이가 1이하인 데이터를 제외
df = df.loc[df['RawText'].str.len() > 1]

In [20]:
# RawText에 중복 데이터가 존재할 수 있으니 중복 제거
df.drop_duplicates(subset='RawText', inplace=True)

In [23]:
# label이 결측치인 데이터는 따로 저장
na_df = df.loc[df['label'].isna(), ]
na_df

,RawText,label
13,귀에서 자꾸 빠져요.귀에 꼽는재 질이 미끄러운 재질이라 작은 소품이지만 재질만 바꾸...,NaN
43,아이를 출산한 기념으로 TV를 바꿨습니다. 그전에 쓰던 TV가 꽤나 무거워서 떨어지...,NaN
44,화면에 노이즈가 생깁니다. 저희 가족 중에 아무도 TV 화면을 건드리거나 하지 않았...,NaN
55,이번에 이사하면서 우리 따님께서 방에 TV가 있으면 좋겠다고 하여 방에서 사용할 T...,NaN
93,기존에 사용하던 무선이어폰이 오래되어서 배터리가 광탈하는 바람에 새로운 상품이 필요...,NaN
...,...,...
41,대용량이고 세척이 간편하다는 얘기에 구매를 했는데 저는 별로인 거 같아요...생각했...,NaN
44,요거 진짜 진짜 물건입니다. 처음엔 디자인이 너무 귀여워서 주문하게 되었는데요. 작...,NaN
68,지인의 추천으로 믿고 바로 구매를 해서 현재도 사용중입니다좀 더 많은 분들에게 도움...,NaN
76,다른 에어쿨러와 다르게 슬림한 디자인이 마음에 들어요.심플한 디자인 덕분에 집안 어...,NaN


In [26]:
# df에는 결측치를 제외
df = df.loc[~df['label'].isna()]
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3678 entries, 0 to 99
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   RawText  3678 non-null   object 
 1   label    3678 non-null   float64
dtypes: float64(1), object(1)
memory usage: 86.2+ KB


In [27]:
df.reset_index(drop=True, inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3678 entries, 0 to 3677
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   RawText  3678 non-null   object 
 1   label    3678 non-null   float64
dtypes: float64(1), object(1)
memory usage: 57.6+ KB


In [28]:
# label의 데이터의 빈도수를 확인
df['label'].value_counts()

label
 1.0    2220
 0.0     944
-1.0     514
Name: count, dtype: int64

In [29]:
# label의 데이터들을 int형태로 변경
df['label'] = df['label'].astype(int)

In [31]:
df['label'].value_counts()

label
 1    2220
 0     944
-1     514
Name: count, dtype: int64

In [33]:
# 삼중 분류의 class를 이진 분류로 변경하기 위해 -1을 0으로 변경
df['label'] = df['label'].map(
    {
        -1 : 0,
        0 : 0,
        1 : 1
    }
)

In [34]:
df['label'].value_counts()

label
1    2220
0    1458
Name: count, dtype: int64

In [35]:
# train, test로 분할
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)

In [36]:
train_df['label'].value_counts()

label
1    1776
0    1166
Name: count, dtype: int64

In [37]:
model_name = 'BM-K/KoSimCSE-roberta-multitask'
sbert = SentenceTransformer(model_name)

No sentence-transformers model found with name BM-K/KoSimCSE-roberta-multitask. Creating a new one with mean pooling.


In [38]:
# 최대 토큰 길이 제한
sbert.max_seq_length = 128

In [40]:
class SBERTHead(Dataset):
    def __init__(self, texts, labels):
        with torch.inference_mode():
            self.emb = sbert.encode(texts, convert_to_tensor=True, normalize_embeddings=True)
        self.labels = torch.tensor(labels, dtype=torch.long)
        # self.texts = texts
        # self.labels2 = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return self.emb[idx], self.labels[idx]
        # getitem에서 임베딩 처리
        # res_emb = sbert.encode(self.texts[idx], convert_to_tensor=True, normalize_embeddings=True)
        # res_label = torch.tensor(self.labels2[idx], dtype=torch.long)
        # return res_emb, res_label

In [41]:
train_ds = SBERTHead(train_df['RawText'].tolist(), train_df['label'].tolist())
test_ds = SBERTHead(test_df['RawText'].tolist(), test_df['label'].tolist())

train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=256)

In [42]:
# 다중 퍼셉트론 구조의 분류 모델 생성
class MLPHead(nn.Module):
    def __init__(self, input_dim, hidden = 256, num_classes = 2, dropout = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_classes)
        )
    def forward(self, x):
        result = self.net(x)
        return result

In [43]:
# sbert모델에서 출력 차원의 개수를 확인
in_dim = sbert.get_sentence_embedding_dimension()
clf = MLPHead(in_dim)
# 손실 함수
crit = nn.CrossEntropyLoss()
# AdamW -> Adam 개선판
opt = optim.AdamW(clf.parameters(), lr=2e-4)

In [44]:
clf.train()

for epoch in range(5):
    total = 0.0
    for x, y in train_dl:
        opt.zero_grad()
        logits = clf(x)
        loss = crit(logits, y)
        loss.backward()
        opt.step()
        total += loss.item() * x.size(0)
    print(f'epoch : {epoch}, loss : {round(total/len(train_df), 4)}')

epoch : 0, loss : 0.6806
epoch : 1, loss : 0.6368
epoch : 2, loss : 0.5919
epoch : 3, loss : 0.5464
epoch : 4, loss : 0.5105


In [48]:
# 정확도, f1_score 확인
clf.eval()

y_true, y_pred = [], []

with torch.inference_mode():
    for x, y in test_dl:
        logits = clf(x)
        pred = logits.argmax(dim=1).tolist()
        y_true += y.tolist()
        y_pred += pred

print('accuracy_score :', accuracy_score(y_true, y_pred))
print('f1_score :', f1_score(y_true, y_pred))

accuracy_score : 0.7866847826086957
f1_score : 0.8356020942408376


In [49]:
samples = na_df['RawText'].sample(10).tolist()
samples

['사용하던 무선 청소기가 흡입력이 안 좋아서 새로 구매하게 됐어요.집에 강아지를 키워서 아무래도 강아지털이 많아서 흡입력이 좋은걸 찾게 되었네요.일단 일주일 정도 사용해보니 생각보다 괜찮은 것 같아요.아주 미친듯이 좋다 그런건 아니지만 대체적으로 마음에 들어요.청소하고 나서 사용시간이 2시간 정도 되서 거실이랑 방 3개는 아주 여유있게 청소하고도 배터리가 남네요. 그 점이 가장 편리하고 저에겐 장점으로 다가왔습니다.그리고 복잡하지 않고 청소기 조작하는게 간단해서 사용하기 편리하네요.손잡이 바로 전면에 보이는 상태 알림 표시가 직관적으로 보여서 청소하면서도 조작하기가 아주 편해요.전자제품이 이렇게 계속 좋아지지 자꾸 더 좋은걸 사고 싶어지네요',
 '출장이 잦아서 구매했는데 생각보다 실망스럽네요.기대하면서 제품 받고 착용했는데 불편했던 점이 꽤나 있었습니다.우선은 버튼들이 모두 측면에 있어서 어디에 어떤 버튼이 있는지가 확실하게 감지가 안되더라구요. 익숙해지면 저절로 손이 가겠지만 초반에는 불편합니다. 그리고 가벼워서 그런지 진동이 전달되는 것이 훨씬 줄어들어서 인식하기가 어려워서 불편합니다. 착용감도 약간 목에서 들떠있는 착용감이라 더욱 진동이 인식이 안되는 것 같아요 엄청 가볍고 사용하기 편리하긴 하지만 동일 제조사의 다른 제품을 먼저 사용하고 있었는데 그 제품이 훨씬 조작하기는 편리한 것 같네요 참고하세요.',
 '제가 남자여도 머리 위쪽이 숲이 적어서 탈모는 아니지만 출근하면서 세련되고 풍성해 보이고 싶어서 이 제품 구매했어요.브러시 형태라서 둥근 빗들고 드라이기로 하려니 미용실처럼 되지는 않아서 힘들었던 점을 보완해 주더라고요.처음에는 그래도 설명서처럼 따라 해서 원하던 스타일을 만들어 내는 게 쉽지 않았어요. 아무리 기계가 좋아도 결국 기술은 필요하긴 하더라고요. 그래도 여러 번 하면서 요령도 붙고 해서 지금은 원하던 데로 가르마 뿌리와 함께 볼륨을 만들어 주고 있어요. 이 정도 볼륨을 만드는 건 그전에도 못했는데 가능해져서 기분이 좋네요.이번에 소

In [55]:
@torch.no_grad()
def predict_review(texts, batch_size=128):
    if isinstance(texts, str):
        texts = [texts]
    texts_norm = [normalize(t) for t in texts]

    sbert.eval()
    clf.eval()

    result = []

    for idx in range(0, len(texts), batch_size):
        batch_text = texts_norm[idx : idx + batch_size]
        embs = sbert.encode(batch_text, convert_to_tensor=True, normalize_embeddings=True)

        logits = clf(embs)
        probs = logits.softmax(dim=-1)
        preds = probs.argmax(dim=-1).tolist()
        id2label = {
            0 : '부정',
            1 : '긍정'
        }
        for idx2, pred in enumerate(preds):
            prob = float(probs[idx2, pred])
            review = texts[idx + idx2]
            label = id2label[pred]
            result.append(
                {
                    'text' : review,
                    'prob' : prob,
                    'label' : label
                }
            )
    return result

In [56]:
predict_review(samples)

[{'text': '사용하던 무선 청소기가 흡입력이 안 좋아서 새로 구매하게 됐어요.집에 강아지를 키워서 아무래도 강아지털이 많아서 흡입력이 좋은걸 찾게 되었네요.일단 일주일 정도 사용해보니 생각보다 괜찮은 것 같아요.아주 미친듯이 좋다 그런건 아니지만 대체적으로 마음에 들어요.청소하고 나서 사용시간이 2시간 정도 되서 거실이랑 방 3개는 아주 여유있게 청소하고도 배터리가 남네요. 그 점이 가장 편리하고 저에겐 장점으로 다가왔습니다.그리고 복잡하지 않고 청소기 조작하는게 간단해서 사용하기 편리하네요.손잡이 바로 전면에 보이는 상태 알림 표시가 직관적으로 보여서 청소하면서도 조작하기가 아주 편해요.전자제품이 이렇게 계속 좋아지지 자꾸 더 좋은걸 사고 싶어지네요',
  'prob': 0.6645014882087708,
  'label': '긍정'},
 {'text': '출장이 잦아서 구매했는데 생각보다 실망스럽네요.기대하면서 제품 받고 착용했는데 불편했던 점이 꽤나 있었습니다.우선은 버튼들이 모두 측면에 있어서 어디에 어떤 버튼이 있는지가 확실하게 감지가 안되더라구요. 익숙해지면 저절로 손이 가겠지만 초반에는 불편합니다. 그리고 가벼워서 그런지 진동이 전달되는 것이 훨씬 줄어들어서 인식하기가 어려워서 불편합니다. 착용감도 약간 목에서 들떠있는 착용감이라 더욱 진동이 인식이 안되는 것 같아요 엄청 가볍고 사용하기 편리하긴 하지만 동일 제조사의 다른 제품을 먼저 사용하고 있었는데 그 제품이 훨씬 조작하기는 편리한 것 같네요 참고하세요.',
  'prob': 0.829912006855011,
  'label': '부정'},
 {'text': '제가 남자여도 머리 위쪽이 숲이 적어서 탈모는 아니지만 출근하면서 세련되고 풍성해 보이고 싶어서 이 제품 구매했어요.브러시 형태라서 둥근 빗들고 드라이기로 하려니 미용실처럼 되지는 않아서 힘들었던 점을 보완해 주더라고요.처음에는 그래도 설명서처럼 따라 해서 원하던 스타일을 만들어 내는 게 쉽지 않았어요. 아무리 기계가 좋아도 결국